$\newcommand{\ket}[1]{\left\lvert #1\right\rangle}
\newcommand{\bra}[1]{\left\langle #1\right\rvert}
\newcommand{\braket}[1]{\left\langle #1\right\rangle}
\newcommand{\proj}[1]{\lvert #1\rangle\!\langle #1\rvert}
\newcommand{\tr}{\operatorname{tr}}\newcommand{\Tr}{\operatorname{Tr}}
\newcommand{\Var}{\operatorname{Var}}\newcommand{\Cov}{\operatorname{Cov}}
\newcommand{\rank}{\operatorname{rank}}\newcommand{\diag}{\operatorname{diag}}
\newcommand{\poly}{\operatorname{poly}}\newcommand{\polylog}{\operatorname{polylog}}
\newcommand{\norm}[1]{\lVert #1\rVert}\newcommand{\abs}[1]{\lvert #1\rvert}
\newcommand{\bigo}{\mathcal{O}}\newcommand{\id}{\mathbb{I}}\newcommand{\dg}{^{\dagger}}
\newcommand{\R}{\mathbb{R}}\newcommand{\C}{\mathbb{C}}\newcommand{\N}{\mathbb{N}}
\newcommand{\Z}{\mathbb{Z}}\newcommand{\E}{\mathbb{E}}\newcommand{\Prob}{\mathbb{P}}
\newcommand{\Hilb}{\mathcal{H}}\newcommand{\ii}{\mathrm{i}}\newcommand{\dd}{\mathrm{d}}
\newcommand{\ee}{\mathrm{e}}\newcommand{\QFT}{\mathrm{QFT}}\newcommand{\CNOT}{\mathrm{CNOT}}
\newcommand{\Uf}{U_f}\newcommand{\defeq}{:=}
\newcommand{\Xgate}{\mathsf{X}}\newcommand{\Ygate}{\mathsf{Y}}
\newcommand{\Zgate}{\mathsf{Z}}\newcommand{\Hgate}{\mathsf{H}}$

# Input Models: State Preparation and Block Encoding

**Learning objectives.**

-   Describe the state-preparation oracle as the *vector* input model and
    state when it can be implemented efficiently.

-   Define block encoding as the *matrix* input model, and read off the
    subnormalization factor $\alpha$, ancilla count $a$, and error $\epsilon$.

-   Apply a block encoding to a prepared state to perform
    *matrix--vector multiplication*, and read its cost off $\alpha$ and the
    output norm.

-   Construct block encodings of $1$-sparse and $s$-sparse matrices from
    entry and location oracles.

-   Distinguish Hermitian from general block encodings and explain why the
    distinction matters for qubitization.

-   Recognize the block-encoding relation in the general projector form
    $A=\alpha\,\Pi'\,U_A\,\Pi$, valid beyond the computational basis.

Part I treated quantum states and unitary circuits as the primitive objects.
Part II turns to *quantum numerical linear algebra*: solving linear systems,
applying matrix functions, simulating dynamics. Every such task begins with the
same question---the *input model*: how does a matrix $A\in\C^{N\times N}$
(with $N=2^n$), which is generally *not* unitary and may be dense, enter a
quantum computer, whose only native operations are unitary? This lecture develops
the two answers that underpin the entire QNLA toolbox. The first, the
*state-preparation oracle*, loads a classical vector into the amplitudes of a
quantum state and is the input model for the right-hand side of a linear system.
The second, *block encoding*, embeds $A$ (rescaled) as the top-left corner of
a larger unitary $U_A$; it is the input model for the matrix itself and the single
abstraction on which linear combination of unitaries (LCU, that chapter),
qubitization and quantum signal processing (QSP, that chapter), the quantum
singular value transformation (QSVT, that chapter), and quantum linear-system
solvers (that chapter) are all built. We give the definition, quantify its cost
with the *subnormalization factor* $\alpha$, and construct block encodings of
sparse matrices explicitly from a pair of oracles---one for entry values, one for
nonzero locations. Throughout, $A$ is an $n$-qubit matrix and
$\norm{A}$ denotes the spectral (operator) norm.

## State-preparation oracles

Many algorithms are posed relative to a fixed input vector: the linear-system
problem $Ax=b$ needs the right-hand side $b$; ground-state energy estimation
(that section) needs an initial guess $\ket{\phi}$; a Monte Carlo estimand is
encoded in a prepared amplitude (that section). In each case the vector must
be loaded into the amplitudes of an $n$-qubit register. This is the vector input
model.

**Definition (State-preparation oracle).**

Let $u\in\C^{N}$ be a nonzero vector with normalized version
$\ket{u}=u/\norm{u}_2=\sum_{i=0}^{N-1}u_i\ket{i}/\norm{u}_2$. A
*state-preparation oracle* for $u$ is an $n$-qubit unitary $O_u$ with
$$O_u\ket{0^n}=\ket{u}.$$
The oracle is *efficient* if $O_u$ is implemented by a circuit of size
$\poly(n)=\polylog(N)$.

Only the direction of $u$ survives normalization; the norm $\norm{u}_2$ is a
classical scalar that must be tracked separately (it reappears, for example, as the
scaling of the solution of a linear system). The essential subtlety is that
the equation above is trivial to *write down* but not always cheap to
*implement*.

**Proposition (Cost of exact amplitude loading).**

For any $\ket{u}\in\C^{N}$ there exists a state-preparation unitary $O_u$
satisfying the equation above using $\bigo(N)=\bigo(2^n)$ elementary gates and no
ancilla qubits. In general this cost cannot be reduced below $\Omega(2^n/n)$ gates
for a worst-case $\ket{u}$.

*Proof (Proof sketch).*

Prepare $\ket{u}$ by a sequence of controlled single-qubit rotations, one per qubit,
that build up the amplitudes conditionally on the bits already set. Concretely, the
first qubit is rotated so that the probabilities of its two branches match the total
weight of $u$ on states with first bit $0$ versus $1$; conditioned on that bit, the
second qubit is rotated to split the weight again; and so on. Level $k$ of this tree
requires $2^{k-1}$ controlled rotations (one per branch), so the total is
$\sum_{k=1}^{n}2^{k-1}=2^{n}-1=\bigo(N)$ rotations, each with phase and magnitude
angles read off from partial sums of $\abs{u_i}^2$ and $\arg u_i$. For the lower
bound, a general $\ket{u}$ carries $\Theta(2^n)$ independent real parameters, while a
circuit of $g$ two-qubit gates on $n$ qubits has only $\bigo(g)$ continuous
parameters; matching the parameter counts forces $g=\Omega(2^n/n)$.

The message of that result is that *arbitrary* amplitude loading is
exponentially expensive and destroys any hope of a quantum speedup on its own.
Efficient state preparation is possible only for *structured* vectors.

**Example (When state preparation is efficient).**

Three broad structures admit $\polylog(N)$-size preparation.

-   *Product (unentangled) states.* If $\ket{u}=\bigotimes_{k=1}^{n}
            \ket{v_k}$ with each $\ket{v_k}$ a single qubit, then $O_u=\bigotimes_k
            R_k$ is $n$ single-qubit rotations. The uniform superposition
    $\Hgate^{\otimes n}\ket{0^n}$ is the special case $\ket{v_k}=\ket{+}$.

-   *Efficiently integrable distributions.* If $\abs{u_i}^2=p(i)$ for a
    probability density $p$ whose cumulative distribution is classically
    computable, the Grover--Rudolph procedure loads $\ket{u}$ with $\bigo(n)$
    controlled rotations, each rotation angle set from an integral of $p$ over a
    dyadic subinterval.

-   *Sparse vectors.* If $u$ has only $d=\poly(n)$ nonzero entries whose
    indices and values are known, $\ket{u}$ can be prepared with $\bigo(dn)$
    gates by loading the $d$ amplitudes one at a time.

When no such structure is available one may fall back on quantum random-access
memory (QRAM), which supplies $O_u$ in $\bigo(n)$ *depth* at the price of
$\bigo(N)$ hardware; whether QRAM is a realistic resource is a standing debate.

We henceforth treat $O_u$ as a given oracle and count *queries* to it, exactly
as we counted queries to $\ee^{-\ii Ht}$ in that chapter. The state-preparation
oracle is the vector half of the input model; the rest of the lecture builds the
matrix half.

## Block encoding: definition and factor

A non-unitary $A$ cannot be realized directly by a circuit. The idea of block
encoding is to place a rescaled copy of $A$ in one *block* of a larger unitary
and to access it by projecting onto that block with ancilla qubits.

### The one-ancilla picture {#the-one-ancilla-picture .unnumbered}

Start with the simplest case. Suppose we can find an $(n+1)$-qubit unitary $U_A$
whose top-left $n$-qubit block equals $A$,
$$U_A=\begin{pmatrix} A & \ast\\ \ast & \ast\end{pmatrix},$$
where $\ast$ denotes blocks irrelevant to the computation. For any $n$-qubit state
$\ket{b}$, feed in the flagged state $\ket{0}\ket{b}$:
$$U_A\ket{0}\ket{b}=\ket{0}\,A\ket{b}+\ket{1}\ket{\perp},$$
for some (unnormalized) state $\ket{\perp}$, so that
$(\bra{0}\otimes\id)\,U_A\,(\ket{0}\otimes\id)=A$. Measuring the ancilla and keeping
the outcome $0$ leaves the system in the normalized state $A\ket{b}/\norm{A\ket{b}}$,
which is obtained with success probability
$$p(0)=\norm{A\ket{b}}^2=\bra{b}A\dg A\ket{b}.$$
Thus the "missing" scalar $\norm{A\ket{b}}$ is recoverable from the measurement
statistics; it depends only on $A$ and $\ket{b}$, not on the irrelevant blocks of
$U_A$. Because a block of a unitary can have operator norm at most $1$, the
embedding the equation above requires $\norm{A}\le1$.

### Rescaling and the general definition {#rescaling-and-the-general-definition .unnumbered}

Most matrices of interest have $\norm{A}>1$, or one simply cannot embed $A$ exactly.
Both obstacles are removed by allowing $U_A$ to encode $A/\alpha$ for a
*subnormalization factor* $\alpha\ge\norm{A}$, using $a$ ancilla qubits, and
tolerating an error $\epsilon$.

**Definition (Block encoding).**

Let $A$ be an $n$-qubit matrix and let $\alpha,\epsilon\ge0$ and $a\in\N$. An
$(a+n)$-qubit unitary $U_A$ is an *$(\alpha,a,\epsilon)$-block-encoding* of $A$
if
$$\norm{\,A-\alpha\,\bigl(\bra{0^a}\otimes\id_n\bigr)\,U_A\,
         \bigl(\ket{0^a}\otimes\id_n\bigr)\,}\le\epsilon.$$
When $\epsilon=0$ the encoding is *exact* and $U_A$ is called an
$(\alpha,a)$-block-encoding. We write $U_A\in\mathrm{BE}_{\alpha,a}(A,\epsilon)$,
and $\mathrm{BE}_{\alpha,a}(A)=\mathrm{BE}_{\alpha,a}(A,0)$.

Equivalently, in the exact case the top-left $n$-qubit block of $U_A$ equals
$A/\alpha$:
$$\bigl(\bra{0^a}\otimes\id_n\bigr)\,U_A\,\bigl(\ket{0^a}\otimes\id_n\bigr)
  =\frac{A}{\alpha},
  \qquad
  U_A=\begin{pmatrix} A/\alpha & \ast\\ \ast & \ast\end{pmatrix},$$
with $2^{a}$ block rows and columns. The three parameters are the currency of
Part II:

-   $\alpha$ (the *subnormalization factor*) rescales $A$ into a block of a
    unitary and so obeys $\alpha\ge\norm{A}$. It is not free: applying $U_A$ to
    $\ket{0^a}\ket{b}$ produces $\ket{0^a}(A/\alpha)\ket{b}+\ket{\perp}$, so the
    flag $\ket{0^a}$ is read with probability
    $\norm{A\ket{b}}^2/\alpha^2$. A larger $\alpha$ shrinks the success
    probability and, more importantly, multiplies the query cost of every
    downstream algorithm (QSVT, QLSP) by $\alpha$. Keeping $\alpha$ close to
    $\norm{A}$ is therefore a central design goal.

-   $a$ is the number of ancilla (flag) qubits; the useful subspace is flagged
    by $\ket{0^a}$.

-   $\epsilon$ is the spectral-norm error, which propagates additively through
    the arithmetic of that chapter.

The verification rule is worth stating separately, as it is used constantly.

**Remark (Verifying a block encoding entrywise).**

$U_A\in\mathrm{BE}_{1,a}(A)$ if and only if
$\bra{0^a,i}U_A\ket{0^a,j}=A_{ij}$ for all $i,j\in[N]$, where
$\ket{0^a,j}\defeq\ket{0^a}\ket{j}$. When $U_A=U_BU_C$ factors, one may compute the
matrix element by splitting the product,
$\bra{0^a,i}U_A\ket{0^a,j}=\bigl(U_B\dg\ket{0^a,i}\bigr)\dg\bigl(U_C\ket{0^a,j}\bigr)$,
evaluating the two states independently and taking their inner product. This is the
workhorse calculation for every construction below.

### Every contraction is block-encodable {#every-contraction-is-block-encodable .unnumbered}

Block encoding is not a restrictive assumption in principle---it is the way any
contraction can be accessed---as the following construction shows.

**Proposition ($(1,1)$-block-encoding is universal for contractions).**

Every $n$-qubit matrix $A$ with $\norm{A}\le1$ admits an exact
$(1,1)$-block-encoding.

*Proof.*

Let $A=W\Sigma V\dg$ be a singular value decomposition, with $W,V$ unitary and
$\Sigma=\diag(\sigma_1,\dots,\sigma_N)$, $\sigma_i\in[0,1]$ since $\norm{A}\le1$.
Because $\sigma_i\le1$, the diagonal matrix $\sqrt{\id-\Sigma^2}$ is real, and the
$(n+1)$-qubit matrix
$$U_A=\begin{pmatrix} W & 0\\ 0 & \id_n\end{pmatrix}
      \begin{pmatrix} \Sigma & \sqrt{\id_n-\Sigma^2}\\
                      \sqrt{\id_n-\Sigma^2} & -\Sigma\end{pmatrix}
      \begin{pmatrix} V\dg & 0\\ 0 & \id_n\end{pmatrix}$$
is unitary: the outer factors are unitary, and the middle factor is unitary because
each $2\times2$ block $\left(\begin{smallmatrix}\sigma_i&\sqrt{1-\sigma_i^2}\\
\sqrt{1-\sigma_i^2}&-\sigma_i\end{smallmatrix}\right)$ is a reflection (its columns
are orthonormal). Its top-left block is $W\Sigma V\dg=A$, so
$U_A\in\mathrm{BE}_{1,1}(A)$.

that result says block encoding costs nothing in principle. The content of
Part II is that for *structured* $A$---sparse matrices (that section),
sums and products of block encodings (that chapter), and matrices arising from
physical Hamiltonians---one can build $U_A$ with a $\polylog(N)$-size circuit and a
small factor $\alpha$, so that the whole pipeline stays efficient.

**Example (Block-encoding a $2\times2$ matrix; reading off $\alpha$).**

Consider the single-qubit matrix
$$A=\begin{pmatrix} 1 & 1\\ 0 & 1\end{pmatrix},
  \qquad \norm{A}=\frac{1+\sqrt5}{2}\approx1.618 ,$$
whose spectral norm is the largest singular value (the golden ratio). Since
$\norm{A}>1$, no exact $(1,\cdot)$-block-encoding exists; take $\alpha=2\ge\norm{A}$
and $A/\alpha=\left(\begin{smallmatrix}1/2&1/2\\0&1/2\end{smallmatrix}\right)$, which
has $\norm{A/\alpha}=\alpha^{-1}\norm{A}\approx0.809<1$. Its SVD gives a
$(1,1)$-block-encoding of $A/\alpha$ by that result; regarded as a
$2$-qubit unitary acting on one flag qubit and one system qubit, this is a
$(2,1)$-block-encoding of $A$. Concretely, with $b=1/2$,
$$U_A=\begin{pmatrix}
        1/2 & 1/2 & \ast & \ast\\
        0   & 1/2 & \ast & \ast\\
        \ast& \ast& \ast & \ast\\
        \ast& \ast& \ast & \ast
      \end{pmatrix},$$
where the starred entries are fixed by demanding orthonormal columns. Applying $U_A$
to $\ket{0}\ket{b}$ and post-selecting on the flag $\ket{0}$ returns
$A\ket{b}/\norm{A\ket{b}}$ with probability $\norm{A\ket{b}}^2/\alpha^2$; had we been
wasteful and chosen $\alpha=10$, the same state would still emerge but the success
probability---and hence the runtime---would drop by a factor $25$. This is the
practical meaning of "keep $\alpha$ close to $\norm{A}$."

## Matrix--vector multiplication: the first primitive

With both input models in hand we can already perform the most basic
quantum numerical-linear-algebra operation: given a matrix $A$ as a block encoding
and a vector $b$ as a state-preparation oracle, produce the normalized
matrix--vector product $A\ket{b}/\norm{A\ket{b}}$. This one primitive is the engine
inside linear solvers, power iteration, and the read-out of a simulated state
$\ee^{-\ii Ht}\ket{b}$.

**Proposition (Applying a block encoding to a state).**

Let $U_A\in\mathrm{BE}_{\alpha,a}(A)$ and let $O_b$ prepare $\ket{b}=O_b\ket{0^n}$.
Then
$$U_A\,\bigl(\id_a\otimes O_b\bigr)\ket{0^a}\ket{0^n}
  =\ket{0^a}\,\frac{A\ket{b}}{\alpha}+\ket{\Phi^{\perp}},
  \qquad \bigl(\proj{0^a}\otimes\id\bigr)\ket{\Phi^{\perp}}=0,$$
and measuring the $a$-qubit flag returns $\ket{0^a}$ with probability
$p=\norm{A\ket{b}}^2/\alpha^2$, upon which the system register collapses to
$A\ket{b}/\norm{A\ket{b}}$.

*Proof.*

$O_b$ produces $\ket{0^a}\ket{b}$; applying $U_A$ and using
$(\bra{0^a}\otimes\id)U_A(\ket{0^a}\otimes\id)=A/\alpha$ (that definition) gives
the stated split, whose flag-$\ket{0^a}$ part is $\ket{0^a}(A/\alpha)\ket{b}$. Its
squared norm $\norm{(A/\alpha)\ket{b}}^2=\norm{A\ket{b}}^2/\alpha^2$ is the
probability of reading $\ket{0^a}$, and the post-measurement system state is the
normalized $A\ket{b}/\norm{A\ket{b}}$.

Three features make this the template for all of Part II.

-   *Input and output share one format.* The result is again an $n$-qubit
    state, like the input $\ket{b}$. Block encoding is thus both an *input*
    and an *output* model: the product feeds straight into another block
    encoding, so matrix--vector products chain into $A_k\cdots A_1\ket{b}$ with no
    read-out of intermediate vectors.

-   *Cost is set by $\alpha$ and the output norm.* One run succeeds with
    probability $p=\norm{A\ket{b}}^2/\alpha^2$, so naive repetition costs
    $\bigo(\alpha^2/\norm{A\ket{b}}^2)$ queries. Amplitude amplification
    (that theorem) removes the square and brings this to
    $\bigo\!\bigl(\alpha/\norm{A\ket{b}}\bigr)$ queries to $U_A$ and $O_b$---the
    quadratic gain that makes the primitive worthwhile, and the quantitative
    reason to keep $\alpha$ small: every matrix--vector product pays for it
    linearly.

-   *Only the direction is produced.* The scalar $\norm{A\ket{b}}$ is not
    carried by the output state; it is recovered from the success probability as
    $\norm{A\ket{b}}=\alpha\sqrt p$, exactly as the norm of a prepared vector had
    to be tracked classically in that section.

**Example (Matrix--vector product with the $2\times2$ encoding).**

Take $A=\left(\begin{smallmatrix}1&1\\0&1\end{smallmatrix}\right)$ with the
$(2,1)$-block-encoding of that example ($\alpha=2$). For $\ket{b}=\ket{0}$
(so $O_b=\id$), $A\ket{0}=\ket{0}$ with $\norm{A\ket{0}}=1$, so the flag is read with
probability $p=1/\alpha^2=1/4$ and the output is $\ket{0}$. For $\ket{b}=\ket{1}$,
$A\ket{1}=\ket{0}+\ket{1}$ has $\norm{A\ket{1}}=\sqrt2$, giving $p=2/4=1/2$ and output
$(\ket{0}+\ket{1})/\sqrt2$. The larger the output norm, the larger the success
probability; and had we used the wasteful $\alpha=10$ of that example, both
probabilities would drop by a factor $25$.

## Block encoding of $s$-sparse matrices

We now give the canonical efficient construction: a block encoding of a sparse
matrix built from two oracles---one that supplies entry *values*, one that
supplies nonzero *locations*. This is the model under which almost all QNLA
complexity is stated.

### The entry and location oracles {#the-entry-and-location-oracles .unnumbered}

Assume $\norm{A}_{\max}\defeq\max_{ij}\abs{A_{ij}}\le1$ (rescale $A$ by
$\norm{A}_{\max}$ otherwise). The *entry oracle* encodes each matrix element in
the amplitude of a signal qubit by a controlled rotation:
$$O_A\ket{0}\ket{i}\ket{j}
   =\Bigl(A_{ij}\ket{0}+\sqrt{1-\abs{A_{ij}}^2}\,\ket{1}\Bigr)\ket{i}\ket{j}.$$
(In practice a database oracle
$\widetilde O_A\ket{0^{d}}\ket{i}\ket{j}=\ket{\tilde A_{ij}}\ket{i}\ket{j}$ returning
a $d$-bit representation of $A_{ij}$ is converted into the equation above by a
controlled rotation followed by uncomputation of the work register.) A matrix is
*$s$-sparse* if every row and column has at most $s$ nonzero entries; the
*location oracle* lists them. For a $1$-sparse matrix each column $j$ has a
single nonzero entry in row $c(j)$, and $c$ is a permutation, so there is a unitary
$$O_c\ket{j}=\ket{c(j)},\qquad O_c\dg\ket{c(j)}=\ket{j}.$$
For general $s$-sparse $A$, write $c(j,\ell)$ for the row of the $\ell$-th nonzero
entry in column $j$ ($\ell\in[s]$, $s=2^{s'}$), and assume a unitary
$$O_c\ket{\ell}\ket{j}=\ket{\ell}\ket{c(j,\ell)}.$$
Reversibility of $O_c$ requires that, given the row $i=c(j,\ell)$ and the index
$\ell$, the column $j$ be recoverable; a banded matrix with
$c(j,\ell)=j+\ell-\ell_0\pmod N$ satisfies this. The entry oracle carries the extra
$\ell$-register along,
$$O_A\ket{0}\ket{\ell}\ket{j}
   =\Bigl(A_{c(j,\ell),j}\ket{0}
     +\sqrt{1-\abs{A_{c(j,\ell),j}}^2}\,\ket{1}\Bigr)\ket{\ell}\ket{j}.$$

### The one-sparse case {#the-one-sparse-case .unnumbered}

**Proposition (Block encoding of a $1$-sparse matrix).**

Let $A$ be $1$-sparse with $\norm{A}_{\max}\le1$, accessed via $O_c$ of
the equation above and $O_A$ of the equation above. Then
$U_A\defeq(\id\otimes O_c)\,O_A$ is a $(1,1)$-block-encoding of $A$.

*Proof.*

Apply $U_A$ to the flagged basis state $\ket{0}\ket{j}$ and take the overlap with
$\ket{0}\ket{i}$:
$$\bra{0}\bra{i}\,U_A\,\ket{0}\ket{j}
  =\bra{0}\bra{i}\Bigl(A_{c(j),j}\ket{0}
     +\sqrt{1-\abs{A_{c(j),j}}^2}\,\ket{1}\Bigr)\ket{c(j)}
  =A_{c(j),j}\,\delta_{i,c(j)}.$$
The $\ket{1}$ branch is annihilated by $\bra{0}$, and $\delta_{i,c(j)}$ selects the
unique nonzero row, so the overlap equals $A_{ij}$ (which is $A_{c(j),j}$ when
$i=c(j)$ and $0$ otherwise). By that remark,
$U_A\in\mathrm{BE}_{1,1}(A)$.

**Example (A $1$-sparse worked encoding: the cyclic shift).**

Let $N=4$ and let $A=S$ be the cyclic shift $S\ket{j}=\ket{(j+1)\bmod4}$, a
permutation matrix; it is $1$-sparse with all nonzero entries equal to $1$, so
$c(j)=(j+1)\bmod4$ and $A_{c(j),j}=1$. The entry oracle the equation above is
trivial here, $O_A\ket{0}\ket{i}\ket{j}=\ket{0}\ket{i}\ket{j}$ (all magnitudes are
$1$, so no rotation onto $\ket{1}$), and $O_c\ket{j}=\ket{(j+1)\bmod4}=S\ket{j}$.
Then $U_A=(\id\otimes O_c)O_A$ acts as $\ket{0}\ket{j}\mapsto\ket{0}\ket{(j+1)
\bmod4}$, and $\bra{0,i}U_A\ket{0,j}=\delta_{i,(j+1)\bmod4}=S_{ij}$. Since $S$ is
already unitary, $U_A$ block-encodes it with $\alpha=1$ and the flag qubit is
inert---a reassuring sanity check that a unitary is its own $(1,0)$-block-encoding
(here padded to one idle ancilla).

**Example (Block encoding of a diagonal matrix).**

For a diagonal $A=\diag(A_{00},\dots,A_{N-1,N-1})$ the location map is the identity,
$c(j)=j$, so $O_c=\id$ and the entry oracle collapses to
$O_A\ket{0}\ket{i}=(A_{ii}\ket{0}+\sqrt{1-\abs{A_{ii}}^2}\ket{1})\ket{i}$. Taking
$U_A=O_A$ gives $\bra{0,i}U_A\ket{0,j}=A_{ii}\delta_{ij}$, so
$U_A\in\mathrm{BE}_{1,1}(A)$: a diagonal matrix is block-encoded by a single
multiply-controlled rotation.

### The general $s$-sparse case {#the-general-s-sparse-case .unnumbered}

For $s>1$ we superpose over the $s$ nonzero locations in each column. Let
$D\defeq\Hgate^{\otimes s'}$ be the *diffusion* operator (the term is inherited
from Grover's search, that chapter), acting on the $s'$-qubit $\ell$-register as
$$D\ket{0^{s'}}=\frac{1}{\sqrt s}\sum_{\ell\in[s]}\ket{\ell}.$$

**Theorem (Block encoding of an $s$-sparse matrix).**

Let $A$ be $s$-sparse with $\norm{A}_{\max}\le1$, accessed via the location oracle
$O_c$ of the equation above and the entry oracle $O_A$ of the equation above.
Define
$$U_A\defeq\bigl(\id\otimes D\otimes\id\bigr)\,
           \bigl(\id\otimes O_c\bigr)\,O_A\,
           \bigl(\id\otimes D\otimes\id\bigr),$$
where the two $D$'s act on the $\ell$-register and the middle $O_c$ writes the row
index. Then $U_A\in\mathrm{BE}_{s,\,s'+1}(A)$: it is an $(s,s'+1)$-block-encoding of
$A$ with subnormalization factor $\alpha=s$.

*Proof.*

The ancilla registers are the signal qubit (call it $\ket{0}$) and the $s'$-qubit
$\ell$-register; the flag is $\ket{0}\ket{0^{s'}}$. Call $\ket{0}\ket{0^{s'}}\ket{j}$
the *source* and $\ket{0}\ket{0^{s'}}\ket{i}$ the *target*. Propagate the
source through $D$, then $O_A$, then $O_c$:
$$\begin{aligned}
  \ket{0}\ket{0^{s'}}\ket{j}
  &\xrightarrow{\ D\ }
    \frac{1}{\sqrt s}\sum_{\ell\in[s]}\ket{0}\ket{\ell}\ket{j}\\
  &\xrightarrow{\ O_A\ }
    \frac{1}{\sqrt s}\sum_{\ell\in[s]}
    \Bigl(A_{c(j,\ell),j}\ket{0}+\sqrt{1-\abs{A_{c(j,\ell),j}}^2}\ket{1}\Bigr)
    \ket{\ell}\ket{j}\\
  &\xrightarrow{\ O_c\ }
    \frac{1}{\sqrt s}\sum_{\ell\in[s]}
    \Bigl(A_{c(j,\ell),j}\ket{0}+\sqrt{1-\abs{A_{c(j,\ell),j}}^2}\ket{1}\Bigr)
    \ket{\ell}\ket{c(j,\ell)}.
\end{aligned}$$
Since only the flag component survives, evolve the target backward, equivalently
apply the (Hermitian) $D$ to it:
$$\ket{0}\ket{0^{s'}}\ket{i}\xrightarrow{\ D\ }
   \frac{1}{\sqrt s}\sum_{\ell'\in[s]}\ket{0}\ket{\ell'}\ket{i}.$$
The block matrix element is the inner product of the two, in which $\bra{0}$ kills
the $\ket{1}$ branch and $\braket{\ell'|\ell}=\delta_{\ell\ell'}$,
$\braket{i|c(j,\ell)}=\delta_{i,c(j,\ell)}$ collapse the double sum:
$$\bra{0}\bra{0^{s'}}\bra{i}\,U_A\,\ket{0}\ket{0^{s'}}\ket{j}
  =\frac{1}{s}\sum_{\ell}A_{c(j,\ell),j}\,\delta_{i,c(j,\ell)}
  =\frac{1}{s}A_{ij},$$
because for each row $i$ there is a unique $\ell$ with $c(j,\ell)=i$ (or the term is
absent, giving $A_{ij}=0$). Thus the top-left block is $A/s$, i.e. $U_A\in\mathrm{BE}_{s,s'+1}(A)$.

The factor $\alpha=s$ is exactly the price of the two diffusion operators: each $D$
spreads a unit amplitude over $s$ locations with weight $1/\sqrt s$, and the two of
them contribute $1/s$ to the surviving block. This is why sparsity controls cost in
QNLA: an $s$-sparse matrix is block-encoded with $\alpha=s$, and downstream query
counts scale linearly in $\alpha$.

**Remark (More general location oracles).**

The single-oracle model the equation above is restrictive: it demands that the row
$c(j,\ell)$ be a fixed function of $(j,\ell)$ with a matching inverse, which is
natural only for banded matrices. A more flexible model uses two oracles,
$O_c\ket{\ell}\ket{j}=\ket{c(j,\ell)}\ket{j}$ and
$O_r\ket{\ell}\ket{i}=\ket{r(i,\ell)}\ket{i}$, listing the $\ell$-th nonzero in
column $j$ and row $i$ respectively; inserting a [swap]{.smallcaps} between them yields
$U_A\in\mathrm{BE}_{s,n+1}(A)$ by the same inner-product bookkeeping (this is the
construction of Gilyén--Su--Low--Wiebe, up to two saved qubits). The value oracle
is correspondingly controlled on both indices,
$O_A\ket{0}\ket{i}\ket{j}=(A_{ij}\ket{0}+\sqrt{1-\abs{A_{ij}}^2}\ket{1})\ket{i}
\ket{j}$; entries with $A_{ij}=0$ never contribute, so the rotation there is
immaterial.

## Hermitian block encoding

The block encoding of that definition constrains only the top-left block of
$U_A$; the unitary itself need not be Hermitian even when $A$ is. Yet several
downstream constructions---most importantly qubitization (that chapter)---are
cleanest when the encoding unitary is *itself* Hermitian.

**Definition (Hermitian block encoding).**

An $(\alpha,a,\epsilon)$-block-encoding $U_A$ of $A$ is a
*$(\alpha,a,\epsilon)$-Hermitian-block-encoding* if $U_A=U_A\dg$. We write
$U_A\in\mathrm{HBE}_{\alpha,a}(A,\epsilon)$, with
$\mathrm{HBE}_{\alpha,a}(A)=\mathrm{HBE}_{\alpha,a}(A,0)$.

Two cautions frame the definition. First, $A$ being Hermitian does *not* make a
given block encoding Hermitian: the $1$-sparse encoding $U_A=(\id\otimes O_c)O_A$ of
that result applied to a Hermitian diagonal $A$ is generally not
self-adjoint (that exercise). Second, a Hermitian $U_A$ that is also unitary
squares to the identity, $U_A^2=\id$, so $U_A$ is a *reflection*; its spectrum
is $\{\pm1\}$. This is precisely the structure qubitization exploits: a Hermitian
block encoding of a Hermitian $A$ makes the pair
$(U_A,\;2\proj{0^a}\otimes\id-\id)$ generate an $\mathrm{SU}(2)$ on each eigenspace of
$A$, turning matrix functions of $A$ into single-qubit rotations. We defer that
machinery to that chapter and here only record how to *build* a Hermitian
encoding for a Hermitian sparse matrix.

**Theorem (Hermitian block encoding of a Hermitian $s$-sparse matrix).**

Let $A$ be Hermitian and $s$-sparse with $\norm{A}_{\max}\le1$. Suppose the
locations are accessed through a single oracle
$O_c\ket{\ell}\ket{j}=\ket{c(j,\ell)}\ket{j}$ (using Hermiticity, the $\ell$-th
nonzero of column $j$ is also the $\ell$-th nonzero of row $j$), and the entries
through a square-root value oracle
$$O_A\ket{i}\ket{0}\ket{j}
   =\ket{i}\Bigl(\sqrt{A_{ij}}\,\ket{0}+\sqrt{1-\abs{A_{ij}}}\,\ket{1}\Bigr)\ket{j},$$
where for complex $A_{ij}=\abs{A_{ij}}\ee^{\ii\theta_{ij}}$ the branch
$\sqrt{A_{ij}}=\sqrt{\abs{A_{ij}}}\,\ee^{\ii\theta_{ij}/2}$ is chosen. Then there is
a circuit built from $D$, $O_c$, $O_A$, a [swap]{.smallcaps}, and their inverses that
realizes $U_A\in\mathrm{HBE}_{s,\,n+2}(A)$.

*Proof (Proof idea).*

Use *two* signal qubits and two index registers, with computational states
$\ket{a}\ket{i}\ket{b}\ket{j}$, $a,b\in\{0,1\}$, and let [swap]{.smallcaps} exchange the
two $(a,i)\leftrightarrow(b,j)$ halves. Propagating the source
$\ket{0}\ket{0^n}\ket{0}\ket{j}$ through $D,O_c,O_A$ deposits a factor
$\sqrt{A_{c(j,\ell),j}}$ on the surviving branch; the [swap]{.smallcaps} then makes the
circuit symmetric under $i\leftrightarrow j$, and propagating the target
$\ket{0}\ket{0^n}\ket{0}\ket{i}$ through the mirror-image gates deposits a matching
$\sqrt{A_{c(i,\ell'),i}}$. Taking the inner product and using Hermiticity
$A_{ij}=A_{ji}^{\ast}$,
$$\bra{0}\bra{0^n}\bra{0}\bra{i}\,U_A\,\ket{0}\ket{0^n}\ket{0}\ket{j}
  =\frac{1}{s}\sqrt{A_{ij}}\sqrt{A_{ji}^{\ast}}
   \sum_{\ell,\ell'}\delta_{i,c(j,\ell)}\delta_{j,c(i,\ell')}
  =\frac{1}{s}A_{ij},$$
since $\sqrt{A_{ij}}\sqrt{A_{ji}^{\ast}}=\sqrt{A_{ij}}\sqrt{A_{ij}}=A_{ij}$ and each
Kronecker delta is satisfied by a unique index. The symmetric placement of $O_A$ and
$O_A\dg$ about the [swap]{.smallcaps} makes the whole circuit equal to its own adjoint, so
$U_A$ is Hermitian; the factor $1/s$ from the two diffusion operators gives
$\alpha=s$, and the register count is $n+2$. This is the Childs--Kothari--Somma
quantum-walk construction.

## Block encoding beyond the computational basis

So far the useful block has been flagged by the *computational* state
$\ket{0^a}$ on the ancilla register. Nothing in the algebra requires this: the flag
can be any subspace, singled out by a projector. This generalization is what lets
block encodings compose (that chapter) and underlies qubitization
(that chapter).

**Definition (Projected (generalized) block encoding).**

Let $U$ be a unitary on a space $\Hilb'\supseteq\Hilb$, and let $\Pi,\Pi'$ be
orthogonal projectors on $\Hilb'$ with images isomorphic to $\Hilb$ via fixed
isometries. Then $U$ is an *$(\alpha,\epsilon)$-projected-block-encoding* of an
operator $A$ on $\Hilb$ if
$$\norm{\,A-\alpha\,\Pi'\,U\,\Pi\,}\le\epsilon,$$
where the norm is taken after identifying $\operatorname{im}\Pi$ and
$\operatorname{im}\Pi'$ with $\Hilb$.

The computational-basis that definition is the special case
$\Pi=\Pi'=\proj{0^a}\otimes\id_n$ and $\Pi'U\Pi=(\ket{0^a}\bra{0^a}\otimes\id)U
(\ket{0^a}\bra{0^a}\otimes\id)$, whose nontrivial block is exactly
$(\bra{0^a}\otimes\id)U(\ket{0^a}\otimes\id)$. Allowing $\Pi\neq\Pi'$ accommodates
rectangular $A$ and asymmetric encodings; allowing a general $\Pi$ lets the flag be
prepared by *any* state-preparation unitary rather than by the bare $\ket{0^a}$.

**Example (LCU flag: a linear combination of unitaries).**

Anticipating that chapter, suppose $A=\sum_{k=0}^{L-1}\beta_k V_k$ with
$\beta_k>0$ and each $V_k$ unitary, and set $\alpha=\sum_k\beta_k$. Let $\mathrm{PREP}$
be the state-preparation unitary (that definition) with
$\mathrm{PREP}\ket{0^a}=\frac{1}{\sqrt\alpha}\sum_k\sqrt{\beta_k}\,\ket{k}$, and let
$\mathrm{SEL}=\sum_k\proj{k}\otimes V_k$ be the multiplexed "select" unitary. Then
$U=(\mathrm{PREP}\dg\otimes\id)\,\mathrm{SEL}\,(\mathrm{PREP}\otimes\id)$ is an
$(\alpha,a)$-block-encoding of $A$, because
$$\bigl(\bra{0^a}\otimes\id\bigr)U\bigl(\ket{0^a}\otimes\id\bigr)
  =\sum_k\bigl(\tfrac{\sqrt{\beta_k}}{\sqrt\alpha}\bigr)^{\!2}\alpha\cdot
   \frac{V_k}{1}
  =\frac{1}{\alpha}\sum_k\beta_k V_k=\frac{A}{\alpha}.$$
Here the flag subspace $\operatorname{im}\Pi$ is $\ket{0^a}$ *as seen through*
$\mathrm{PREP}$; the "beyond the computational basis" viewpoint of
that definition makes this manifest, since $\Pi=\mathrm{PREP}\proj{0^a}
\mathrm{PREP}\dg$ projects onto the prepared coefficient state. This LCU block
encoding, with subnormalization $\alpha=\sum_k\beta_k$ the $\ell_1$-norm of the
coefficients, is the second pillar of QNLA and the subject of the next lecture.

**Remark (Why the projector form is the right abstraction).**

The generalized form the equation above exposes what block encoding really
is: a unitary $U$ together with a way of "entering" ($\Pi$) and "exiting"
($\Pi'$) a distinguished subspace on which $U$ acts as $A/\alpha$. Products of block
encodings multiply the operators (with the projectors composing), sums combine
through an LCU flag, and---crucially for that chapter---the singular
values of $A$ become the parameters of the $\mathrm{SU}(2)$ rotations that
$U,\Pi,\Pi'$ generate on the two-dimensional invariant subspaces. Every algorithm in
Part II is, at bottom, a statement about polynomials of the operator $\Pi'U\Pi$.

## Exercises {#exercises .unnumbered}

**Exercise.**

*Norm constraints on a block.* Let $U_A$ be an exact $(\alpha,a)$-block-encoding
of $A$. Show that $\norm{A}\le\alpha$, so that $\alpha\ge\norm{A}$ is necessary.
Conversely exhibit, for any $A$ with $\norm{A}\le\alpha$, a $(1,\cdot)$-block-encoding
of $A/\alpha$ and hence an $(\alpha,\cdot)$-block-encoding of $A$
(cf. that result). Explain why $\norm{A}_{\max}\le1$ does *not* by
itself imply $\norm{A}\le1$, and give a $2\times2$ example where
$\norm{A}=s\,\norm{A}_{\max}$ for $s=2$.

**Exercise.**

*Success probability and subnormalization.* Let $U_A$ be an
$(\alpha,a)$-block-encoding of $A$ and $\ket{b}$ an $n$-qubit state. Show that
applying $U_A$ to $\ket{0^a}\ket{b}$ and measuring the flag register yields $\ket{0^a}$
with probability $p=\norm{A\ket{b}}^2/\alpha^2$, and that conditioned on this outcome
the system register holds $A\ket{b}/\norm{A\ket{b}}$. Deduce how the expected number
of repetitions needed to see the flag once scales with $\alpha$, and comment on the
practical cost of an overly large $\alpha$ (cf. that example).

**Exercise.**

*Verifying a $1$-sparse encoding.* Let $A$ be the $4\times4$ matrix with
$A_{c(j),j}=a_j$ in row $c(j)=(j+1)\bmod4$ and zeros elsewhere, where
$a_0,\dots,a_3\in[0,1]$. Write down the oracles $O_c$ and $O_A$ of
the equation above--the equation above, form $U_A=(\id\otimes O_c)O_A$, and verify
directly that $\bra{0,i}U_A\ket{0,j}=A_{ij}$ for all $i,j$, confirming
$U_A\in\mathrm{BE}_{1,1}(A)$.

**Exercise.**

*A Hermitian matrix with a non-Hermitian encoding.* Take the Hermitian diagonal
matrix $A=\diag(a_0,a_1)$ with $a_0\neq a_1$ in $[0,1]$, and its $(1,1)$-block-encoding
$U_A=O_A$ from that example (a controlled rotation). Write $U_A$ as an
explicit $4\times4$ matrix and show that $U_A\neq U_A\dg$ in general, even though $A$
is Hermitian. Explain in one sentence why this motivates the separate notion of a
Hermitian block encoding (that definition).

**Exercise.**

*The sparsity factor $\alpha=s$.* In the proof of that theorem, track
where each factor of $1/\sqrt s$ arises. Show that the two diffusion operators are
jointly responsible for the block being $A/s$ rather than $A$, and conclude that any
construction based on the equation above has subnormalization $\alpha=s$. What is the
block-encoding factor of a dense matrix ($s=N$) built this way, and why does this make
the construction useless for unstructured dense matrices?

**Exercise.**

*An LCU block encoding of a Pauli sum.* Let $A=\tfrac12(\Xgate+\Zgate)$ on a
single qubit. Using that example with $L=2$, $\beta_0=\beta_1=\tfrac12$,
$V_0=\Xgate$, $V_1=\Zgate$, write down $\mathrm{PREP}$ (a single-qubit rotation on the
one flag qubit), $\mathrm{SEL}$ (a two-qubit controlled gate applying $\Xgate$ or
$\Zgate$), and verify that
$U=(\mathrm{PREP}\dg\otimes\id)\mathrm{SEL}(\mathrm{PREP}\otimes\id)$ is a
$(1,1)$-block-encoding of $A$. What is $\norm{A}$, and is $\alpha=1$ optimal here?

**Exercise.**

*Encoding a Hermitian matrix on the unit circle.* Let $A\in\C^{N\times N}$
($N=2^n$) be Hermitian with entries on the complex unit circle,
$A_{ij}=\ee^{\ii\theta_{ij}}$, $\theta_{ij}\in[0,2\pi)$, accessible through a
$2n$-qubit unitary $V$ with
$$V\ket{0^n}\ket{j}=\frac{1}{\sqrt N}\sum_{i\in[N]}\ee^{\ii\theta_{ij}/2}\ket{i}\ket{j},
  \qquad j\in[N].$$
Use $V$ to build a block encoding $U$ of $A$ with $n$ ancilla qubits (hint: the
target register should be processed by a mirror-image copy of $V$, exploiting
$\theta_{ji}=-\theta_{ij}$ from Hermiticity), and determine the subnormalization
factor $\alpha$.

## Further reading {#further-reading .unnumbered}

-   Lin Lin, *qasc notes*, Ch. 6 (query model for matrix entries, block
    encoding, $s$-sparse and Hermitian block encodings, general query models).

-   Lin Lin, *live notes*, Ch. 9 (block encoding, the LCU introduction,
    block encodings of matrix sums and products, and block encoding beyond the
    computational basis).

-   Dong An, *Quantum Numerical Linear Algebra* tutorial talk (the vector
    and matrix input-model frames and the block-encoding definition).

-   A. Gilyén, Y. Su, G. H. Low, and N. Wiebe, *Quantum singular value
    transformation and beyond*, STOC 2019 (Lemma 48, the sparse-matrix
    block-encoding construction).